In [ ]:
# libraries
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# modules
from src.config import project_path
from src.core import Participant

def get_exercise_and_condition(exercise_num: int) -> tuple:

    if exercise_num == 1:
        ex_id: str = 'FingerTapping'
        condition_id: str = 'Affected'
    elif exercise_num == 2:
        ex_id: str = 'FingerTapping'
        condition_id: str = 'Healthy'
    elif exercise_num == 3:
        ex_id: str = 'FingerAlternation'
        condition_id: str = 'Affected'
    elif exercise_num == 4:
        ex_id: str = 'FingerAlternation'
        condition_id: str = 'Healthy'
    elif exercise_num == 5:
        ex_id: str = 'HandOpening'
        condition_id: str = 'Affected'
    elif exercise_num == 6:
        ex_id: str = 'HandOpening'
        condition_id: str = 'Healthy'
    elif EXERCISE_NUM == 7:
        ex_id: str = 'ProSup'
        condition_id: str = 'Affected'
    elif exercise_num == 8:
        ex_id: str = 'ProSup'
        condition_id: str = 'Healthy'

    return ex_id, condition_id

# set variables to a specific exercise
P_ID: str = 'P027'
VISIT_ID: str = 'T2'
EXERCISE_NUM: int = 6

EX_ID, CONDITION_ID = get_exercise_and_condition(EXERCISE_NUM)

# define paths
PICKLE_PATH = os.path.join(project_path, 'data', '03_processed', f'{P_ID}_{VISIT_ID}.pickle')
REGISTRY_PATH = os.path.join(project_path, 'data', '03_processed', 'trim_registry.csv')

# load data from pickle
if not os.path.exists(PICKLE_PATH):
    print(f'Error: Pickle file not found at {PICKLE_PATH}')
else:
    p = Participant.load(PICKLE_PATH)
    ex_key = f'{EX_ID}_{CONDITION_ID}'

    if ex_key not in p.exercises:
        print(f'Error: Exercise "{ex_key}" not found for {P_ID}.')
    else:
        ex = p.exercises[ex_key]
        metrics = getattr(ex, 'metrics', {})

        # extract the time (t) and signal (y) arrays based on the metric keys
        t_data, y_data = None, None

        if 'FingerTapping' in EX_ID:
            t_data = metrics.get('idx_tap_time_series_x')
            y_data = metrics.get('idx_tap_time_series_y')
        elif 'FingerAlternation' in EX_ID:
            t_data = metrics.get('alt_tap_time_series_x')
            y_data = metrics.get('alt_tap_time_series_y')
        elif 'HandOpening' in EX_ID:
            t_data = metrics.get('open_close_time_series_x')
            y_data = metrics.get('open_close_time_series_y')
        elif 'ProSup' in EX_ID:
            t_data = metrics.get('pro_sup_time_series_x')
            y_data = metrics.get('pro_sup_time_series_y')

        if t_data is None or len(t_data) == 0:
            print(f'Error: No time series metrics found for {ex_key}. Has 04_extract_kinematics run?')
        else:
            print(f'Successfully loaded {ex_key} for {P_ID}')

            # render interactive plotly
            fig = go.FigureWidget()

            # handle single arrays (tapping) and list of arrays (finger alternation)
            if isinstance(y_data, list) and isinstance(y_data[0], (list, np.ndarray)):
                # multiple signals (e.g., 4 fingers)
                primary_t = np.array(t_data[0])
                for i, (t_arr, y_arr) in enumerate(zip(t_data, y_data)):
                    fig.add_trace(go.Scatter(x=t_arr, y=y_arr, mode='lines', name=f'Signal {i+1}', opacity=0.8))
            else:
                # single signal
                primary_t = np.array(t_data)
                fig.add_trace(go.Scatter(x=primary_t, y=y_data, mode='lines', name='Distance/Angle', line=dict(color='blue')))

            fig.update_layout(
                title=f'Trimming Tool: {P_ID} | {VISIT_ID} | {ex_key}',
                xaxis=dict(title='Time [s]', rangeslider=dict(visible=True, thickness=0.15)),
                yaxis=dict(title='Amplitude'),
                height=500,
                template='plotly_white'
            )

            # create save button
            save_btn = widgets.Button(description='Save Trim to Registry', button_style='success', layout={'width': '300px'})
            output_log = widgets.Output()

            # define save logic
            def on_save_clicked(b):
                with output_log:
                    output_log.clear_output()

                    # get the time limits from the slider
                    x_range = fig.layout.xaxis.range

                    # if slider wasn't moved, plotly returns None. Default to full array.
                    if x_range is None:
                        start_time, end_time = primary_t[0], primary_t[-1]
                    else:
                        start_time, end_time = x_range[0], x_range[1]

                    # translate time [s] back to exact frame indices
                    start_frame = int(np.searchsorted(primary_t, start_time))
                    end_frame = int(np.searchsorted(primary_t, end_time))

                    # boundary safety
                    start_frame = max(0, start_frame)
                    end_frame = min(len(primary_t) - 1, end_frame)

                    # update the CSV registry
                    if not os.path.exists(REGISTRY_PATH):
                        print(f'Error: Registry not found at {REGISTRY_PATH}')
                        return

                    reg_df = pd.read_csv(REGISTRY_PATH)

                    # match exact row using metadata from the Exercise object
                    mask = (
                            (reg_df['p_ID'] == P_ID) &
                            (reg_df['visit_ID'] == VISIT_ID) &
                            (reg_df['ex_name'] == ex.exercise_id) &
                            (reg_df['side_focus'] == ex.side_focus)
                    )

                    if not mask.any():
                        print(f'Error: Could not find matching row in registry for this exercise.')
                    else:
                        reg_df.loc[mask, 'start_frame'] = start_frame
                        reg_df.loc[mask, 'end_frame'] = end_frame
                        reg_df.to_csv(REGISTRY_PATH, index=False)

                        print(f'Success! Registry updated.')
                        print(f'Slider Time : [{start_time:.2f}s  -  {end_time:.2f}s]')
                        print(f'Frame Index : [{start_frame}  -  {end_frame}]')
                        print("Next step: Delete this participant's Pickle file and rerun 02_load_participants.")

            save_btn.on_click(on_save_clicked)

            # display the UI
            display(fig)
            display(save_btn, output_log)